# NYSgpt rerankers — usage

Subscribe to the listing on AWS Marketplace, copy the model package ARN for your Region from the subscription's configuration page, then run the cells.

In [ ]:
import json, boto3, sagemaker
from sagemaker import ModelPackage, get_execution_role

session = sagemaker.Session()
role = get_execution_role()
model_package_arn = "<model package ARN from your AWS Marketplace subscription>"

## Real-time endpoint

In [ ]:
model = ModelPackage(role=role, model_package_arn=model_package_arn, sagemaker_session=session)
predictor = model.deploy(initial_instance_count=1, instance_type="ml.m5.xlarge", endpoint_name="nysgpt-reranker")

In [ ]:
runtime = boto3.client("sagemaker-runtime")
payload = json.load(open("data/sample-input.json"))
resp = runtime.invoke_endpoint(EndpointName="nysgpt-reranker", ContentType="application/json", Body=json.dumps(payload))
print(json.loads(resp["Body"].read())["ranking"])

## Batch transform

In [ ]:
bucket = session.default_bucket()
input_uri = session.upload_data("data/sample-input.json", bucket=bucket, key_prefix="nysgpt-reranker/input")
transformer = model.transformer(instance_count=1, instance_type="ml.m5.xlarge", output_path=f"s3://{bucket}/nysgpt-reranker/output")
transformer.transform(input_uri, content_type="application/json")
transformer.wait()

## Clean up

In [ ]:
predictor.delete_endpoint()
model.delete_model()